In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

In [57]:
df = pd.read_csv('BasicSurvey(3-6-20).csv', low_memory=False)
pattern = rf"^({'usedrugs'})_[1-8]$"
tv_cols = df.columns[df.columns.str.match(pattern)].tolist()

df[tv_cols]

,usedrugs_1,usedrugs_2,usedrugs_3,usedrugs_4,usedrugs_5,usedrugs_6,usedrugs_7,usedrugs_8
0,Not at all,Not at all,Not at all,Not at all,Not at all,Not at all,Not at all,NaN
1,Not at all,Not at all,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Less than 1-2 times a month,NaN,NaN,Not at all,NaN,NaN,NaN
3,NaN,Not at all,Not at all,NaN,NaN,NaN,NaN,NaN
4,NaN,Not at all,Not at all,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
717,NaN,Not at all,NaN,Not at all,NaN,Not at all,NaN,NaN
718,NaN,Not at all,NaN,NaN,NaN,NaN,NaN,NaN
719,Not at all,Not at all,NaN,NaN,NaN,NaN,NaN,NaN
720,Not at all,Not at all,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
assert df["egoid"].is_unique,"Duplicate participants rows detected!"
assert df['egoid'].notna().all(), "Null primary keys detected!"


In [46]:
bases = df.columns.to_series().str.replace(r"_\d+$", "", regex=True)

schema = (
    pd.DataFrame({
        "col": df.columns,
        "base":bases.values,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values
    })
    .groupby("base", as_index=False)
    .agg(
        n_waves=("col","count"),
        dtypes=("dtype", lambda s: ",".join(sorted(s.unique()))),
        total_non_null = ("non_null","sum")
    )
    .sort_values(["n_waves","total_non_null"], ascending=[False,False])
)


In [47]:
schema

,base,n_waves,dtypes,total_non_null
819,physicalactivity1,8,float64,3897
722,majorevent,8,object,3088
929,usecaffine,8,object,3083
498,clubsports,8,object,3082
931,usedrugs_prescr,8,object,3082
...,...,...,...,...
665,hawaii1,1,float64,0
667,hawaii3,1,float64,0
899,studyAbroadF_6,1,float64,0
907,studyAbroad_4,1,float64,0


In [54]:
#schema.query("n_waves >= 7 & " )
#schema[schema["dtypes"].str.contains(",")]
schema["n_waves"].value_counts().sort_index()
schema = schema.query("total_non_null >= 2000" )

schema.to_csv("possible_categories.csv")

In [ ]:

meta_cols = ['egoid', 'Tier', 'gender_1']
date_cols = [f"{p}_{w}" for w in range(1, 9) for p in ['StartDate', 'EndDate']]
fb_cols = ['fbaccuracy_2', 'fbaccuracy_3', 'fbaccuracy_8']

tv_bases = [
    'Health', 'Happy', 'Weight', 'Height', 'Dieting',
    'sedentarytime_scale', 'PSQI_duration',
    'usetobacco', 'usecaffine', 'usebeer', 'usewine', 'usedrugs', 'usedrugs_prescr'
]
tv_cols = [f"{b}_{w}" for b in tv_bases for w in range(1, 9) if f"{b}_{w}" in df.columns]
ex_cols = ['exercise_1'] + [f"{b}_{w}" for w in range(2, 9) for b in ['exercisealone', 'exerciseothers']]

selected_cols = meta_cols + date_cols + fb_cols + tv_cols + ex_cols
assert len(selected_cols) == len(set(selected_cols)), "Duplicate columns!"
assert all(c in df.columns for c in selected_cols), "Missing columns!"

df_clean = df[selected_cols].copy()
out_path = '/mnt/data/BasicSurvey_cleaned_139cols.csv'
df_clean.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", df_clean.shape)
print("\nSample unique values for ordinal/binary columns (Wave 1 or 2):")
for col in ['Health_1', 'Happy_1', 'Dieting_1', 'usetobacco_1', 'usecaffine_1', 'usebeer_1',
            'usewine_1', 'usedrugs_1', 'usedrugs_prescr_1', 'exercise_1', 'exercisealone_2', 'exerciseothers_2']:
    print(f"  {col:20s}: {df_clean[col].dropna().unique().tolist()}")

In [ ]:



df = pd.read_csv("BasicSurvey_cleaned_139cols.csv")

# 1. Fill missing Height within each person (across waves 1..8) and compute BMI (lbs/in^2 * 703)
h_cols = [f"Height_{w}" for w in range(1, 9) if f"Height_{w}" in df.columns]
height_filled = df[h_cols].ffill(axis=1).bfill(axis=1).median(axis=1)

for w in range(1, 9):
    df[f"BMI_{w}"] = (df[f"Weight_{w}"] / (height_filled ** 2)) * 703

# 2. Map ordinal frequency strings to numeric levels (including '0' in exercise_1)
freq_map = {
    "0": 0,
    "Not at all": 0,
    "Less than 1-2 times a month": 1,
    "1-2 times a month": 2,
    "1-2 times a week": 3,
    "Three times a week or more": 4,
}

# 3. Harmonize Exercise across Waves 1..8
df["exercise_1_num"] = df["exercise_1"].map(freq_map)
for w in range(2, 9):
    alone = df[f"exercisealone_{w}"].map(freq_map)
    others = df[f"exerciseothers_{w}"].map(freq_map)
    df[f"exercise_{w}_num"] = np.fmax(alone, others)  # preserves NaN only if both are NaN

# 4. Merge Alcohol (beer + wine) and Drugs (illicit + prescription)
for w in range(1, 9):
    df[f"use_alcohol_{w}"] = np.fmax(df[f"usebeer_{w}"].map(freq_map), df[f"usewine_{w}"].map(freq_map))
    df[f"use_drugs_{w}"] = np.fmax(df[f"usedrugs_{w}"].map(freq_map), df[f"usedrugs_prescr_{w}"].map(freq_map))

In [ ]:
import re
import pandas as pd

meta_cols = ["egoid", "Tier", "gender_1"]
date_cols = df.columns[df.columns.str.match(r"^(StartDate|EndDate)_[1-8]$")].tolist()
fb_cols = df.columns[df.columns.str.match(r"^fbaccuracy_[1-8]$")].tolist()

target_bases = [
    "Health", "Happy", "Weight", "Height", "Dieting",
    "sedentarytime_scale", "PSQI_duration",
    "usetobacco", "usecaffine", "usebeer", "usewine",
    "usedrugs", "usedrugs_prescr", "exercise", "exercisealone", "exerciseothers"
]

pattern = rf"^({'|'.join(target_bases)})_[1-8]$"
tv_cols = df.columns[df.columns.str.match(pattern)].tolist()

df_sub = df[meta_cols + date_cols + fb_cols + tv_cols].copy()